# Module 04: Creating and Managing Stimuli/Events Files

Task-based fMRI experiments record **when** stimuli were presented and **what** participants
experienced.  The BIDS standard captures this information in plain-text **events TSV files**
that accompany each BOLD run.  Without accurate events files you cannot build a first-level
GLM model, so getting them right is essential.

## What you will learn
1. The structure and required columns of a BIDS events file.
2. How to load and explore events files with pandas.
3. How to simulate and parse a PsychoPy-style log file.
4. How to convert raw behavioural data to BIDS TSV format.
5. How to validate events files.
6. How to visualise trial structure.
7. How to compute per-condition summary statistics.

**Prerequisites:** Modules 00-03 completed.

**Estimated time:** 2 hours.

## 1  BIDS Events File Format

Each task-fMRI BOLD run has a companion TSV file following this naming pattern:

```
sub-<label>_task-<label>[_run-<index>]_events.tsv
```

### Required columns

| Column | Type | Description |
|--------|------|-------------|
| `onset` | float (s) | Onset time relative to the start of the BOLD run |
| `duration` | float (s) | Duration of the event |
| `trial_type` | string | Experimental condition label |

### Recommended columns

| Column | Type | Description |
|--------|------|-------------|
| `response_time` | float (s) | Reaction time; use `n/a` if no response |
| `stim_file` | string | Path to stimulus file |
| `HED` | string | Hierarchical Event Descriptor tags |

> **Important:** Missing numeric values must be written as `n/a` (not `NaN`, `None`, or empty).

## 2  The Emotion Regulation Paradigm

The sample dataset uses an **emotion regulation** task with three conditions:

| Condition | `trial_type` | Instruction |
|-----------|-------------|-------------|
| Reappraise | `Reappraise` | Cognitively reappraise the negative image |
| Look | `Look` | Passively view the negative image |
| Suppress | `Suppress` | Suppress expressive responses |

### Trial structure (one trial ≈ 18 s)

```
│ Fixation (2 s) │ Image + Instruction (12 s) │ Rating (4 s) │
```

Each run contains **~24 trials** (8 per condition), resulting in a run duration of
approximately 8 minutes.

In [ ]:
# Cell 3: Imports and setup
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ensure the repo root is on the path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Matplotlib style
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

## 3  Loading and Exploring an Existing Events File

We will load one of the events files from the sample dataset.
If you have run the data download in Module 01, the file should be present;
otherwise we create a minimal example in the next cell.

In [ ]:
# Cell 4: Load (or create) a sample events file
BIDS_ROOT = os.path.join(REPO_ROOT, 'data', 'bids_dataset')
events_path = os.path.join(
    BIDS_ROOT,
    'sub-01', 'func',
    'sub-01_task-emotionreg_run-1_events.tsv'
)

if os.path.isfile(events_path):
    print(f'Loading existing events file:\n  {events_path}')
    events = pd.read_csv(events_path, sep='\t', na_values=['n/a'])
else:
    print('Sample events file not found — creating a synthetic example.')
    rng = np.random.default_rng(42)
    conditions = ['Reappraise', 'Look', 'Suppress']
    trial_types = conditions * 8  # 8 trials per condition = 24 trials
    rng.shuffle(trial_types)
    onsets = np.cumsum([0] + [18.0] * (len(trial_types) - 1))  # one trial every 18 s
    durations = np.full(len(trial_types), 12.0)
    rts = rng.uniform(0.4, 2.5, size=len(trial_types))
    rts[rng.random(len(trial_types)) < 0.1] = np.nan  # ~10 % missed responses
    events = pd.DataFrame({
        'onset': onsets,
        'duration': durations,
        'trial_type': trial_types,
        'response_time': np.round(rts, 3),
    })

print(f'\nShape: {events.shape}')
events.head(8)

In [ ]:
# Cell 5: Basic DataFrame exploration
print('=== Column dtypes ===')
print(events.dtypes)

print('\n=== Descriptive statistics ===')
print(events.describe())

print('\n=== Trial type counts ===')
print(events['trial_type'].value_counts())

print(f"\nRun duration (end of last trial): {events['onset'].iloc[-1] + events['duration'].iloc[-1]:.1f} s")

## 4  Creating Events from Raw Behavioural Log Data

In practice, your stimulus software (PsychoPy, E-Prime, Presentation, etc.) saves a
run log file with experiment-specific column names.  You need to:

1. Parse the raw log.
2. Map columns to BIDS names.
3. Compute onset times relative to the scanner trigger.
4. Write the TSV.

Below we simulate a minimal PsychoPy-style CSV output and then convert it.

In [ ]:
# Cell 6: Simulate a PsychoPy-style CSV log
rng = np.random.default_rng(99)
conditions = ['Reappraise', 'Look', 'Suppress']
n_trials = 24
trial_types = (conditions * (n_trials // len(conditions)))
rng.shuffle(trial_types)

# PsychoPy timestamps are relative to experiment start, not scanner trigger
scanner_trigger_time = 6.0  # dummy scan period = 6 s
trial_onsets_experiment = scanner_trigger_time + np.cumsum(
    [0] + [18.0] * (n_trials - 1)
)

raw_log = pd.DataFrame({
    'condition':       trial_types,
    'image_onset_exp': trial_onsets_experiment,          # PsychoPy time
    'image_dur':       np.full(n_trials, 12.0),
    'key_resp.rt':     np.where(
        rng.random(n_trials) < 0.1,
        None,
        np.round(rng.uniform(0.4, 2.5, n_trials), 3)
    ),
    'key_resp.corr':   rng.integers(0, 2, n_trials),
})

print('Raw PsychoPy log (first 5 rows):')
raw_log.head()

## 5  Converting to BIDS TSV Format

In [ ]:
# Cell 7: Convert raw log to BIDS events
def psychopy_to_bids_events(raw_df, scanner_trigger_s):
    """
    Convert a PsychoPy-style DataFrame to a BIDS events DataFrame.

    Parameters
    ----------
    raw_df : pd.DataFrame
        Raw log with PsychoPy column names.
    scanner_trigger_s : float
        Time of the first scanner trigger in PsychoPy clock units (seconds).

    Returns
    -------
    pd.DataFrame
        BIDS-compliant events DataFrame.
    """
    bids = pd.DataFrame()

    # Onset is relative to the scanner trigger
    bids['onset'] = (raw_df['image_onset_exp'] - scanner_trigger_s).round(3)
    bids['duration'] = raw_df['image_dur'].round(3)
    bids['trial_type'] = raw_df['condition']

    # Response time: replace None/NaN with np.nan; keep as float
    rt = pd.to_numeric(raw_df['key_resp.rt'], errors='coerce')
    bids['response_time'] = rt.where(rt > 0).round(3)

    # Optional: pass-through additional columns
    bids['correct'] = raw_df['key_resp.corr']

    return bids.sort_values('onset').reset_index(drop=True)


bids_events = psychopy_to_bids_events(raw_log, scanner_trigger_time)
print('BIDS events (first 5 rows):')
print(bids_events.head())
print(f'\nTotal trials : {len(bids_events)}')
print(f'Missed RTs   : {bids_events["response_time"].isna().sum()}')

In [ ]:
# Cell 8: Write BIDS TSV to disk
import tempfile

out_dir = tempfile.mkdtemp(prefix='bids_events_tutorial_')
out_path = os.path.join(out_dir, 'sub-01_task-emotionreg_run-1_events.tsv')

# BIDS requires 'n/a' for missing values, not empty cells
bids_events.to_csv(out_path, sep='\t', index=False, na_rep='n/a')
print(f'Written to: {out_path}')

# Verify round-trip
reloaded = pd.read_csv(out_path, sep='\t', na_values=['n/a'])
print('\nReloaded from disk:')
reloaded.head()

## 6  Validating Events Files

Before proceeding to GLM modelling, verify:

- Required columns are present.
- All onsets are ≥ 0.
- All durations are > 0.
- No trials overlap in time.

In [ ]:
# Cell 9: Inline validation function
def validate_events(df, run_duration_s=None):
    """
    Validate a BIDS events DataFrame.

    Returns a dict with keys 'passed' (bool) and 'issues' (list of str).
    """
    issues = []
    required = ['onset', 'duration', 'trial_type']

    # Required columns
    for col in required:
        if col not in df.columns:
            issues.append(f'[ERROR] Missing required column: {col}')

    if 'onset' in df.columns:
        if (df['onset'] < 0).any():
            issues.append('[ERROR] Negative onset values found.')

    if 'duration' in df.columns:
        if (df['duration'] <= 0).any():
            issues.append('[ERROR] Non-positive duration values found.')

    # Overlap check
    if {'onset', 'duration'}.issubset(df.columns):
        sorted_df = df.sort_values('onset')
        ends = sorted_df['onset'].values + sorted_df['duration'].values
        next_onsets = sorted_df['onset'].values[1:]
        n_overlaps = (ends[:-1] > next_onsets + 1e-6).sum()
        if n_overlaps:
            issues.append(f'[ERROR] {n_overlaps} overlapping trial pair(s).')

    # Optional: run duration check
    if run_duration_s is not None and {'onset', 'duration'}.issubset(df.columns):
        last_end = (df['onset'] + df['duration']).max()
        if last_end > run_duration_s + 1.0:
            issues.append(
                f'[ERROR] Last event ends at {last_end:.1f}s '
                f'but run duration is {run_duration_s:.1f}s.'
            )

    return {
        'passed': all('[ERROR]' not in i for i in issues),
        'issues': issues,
    }


result = validate_events(bids_events, run_duration_s=450.0)
print(f"Validation passed: {result['passed']}")
if result['issues']:
    for issue in result['issues']:
        print(f'  {issue}')
else:
    print('  No issues found.')

## 7  Visualising Trial Structure

A **timeline plot** (Gantt-style) makes it easy to visually inspect the trial
sequence, inter-trial intervals, and condition balance.

In [ ]:
# Cell 10: Trial structure timeline plot
CONDITION_COLORS = {
    'Reappraise': '#4C72B0',
    'Look':       '#DD8452',
    'Suppress':   '#55A868',
}

def plot_trial_timeline(events_df, title='Trial Structure Timeline'):
    """
    Draw a Gantt-style timeline of trial onsets and durations.

    Parameters
    ----------
    events_df : pd.DataFrame
        BIDS events DataFrame with onset, duration, trial_type columns.
    title : str
        Plot title.
    """
    fig, ax = plt.subplots(figsize=(14, 3))

    conditions = sorted(events_df['trial_type'].unique())
    default_colors = plt.cm.tab10.colors
    color_map = {
        c: CONDITION_COLORS.get(c, default_colors[i % 10])
        for i, c in enumerate(conditions)
    }

    for _, row in events_df.iterrows():
        ax.barh(
            y=0,
            width=row['duration'],
            left=row['onset'],
            height=0.6,
            color=color_map[row['trial_type']],
            edgecolor='white',
            linewidth=0.5,
            alpha=0.85,
        )

    legend_patches = [
        mpatches.Patch(color=color_map[c], label=c) for c in conditions
    ]
    ax.legend(handles=legend_patches, loc='upper right', framealpha=0.9)
    ax.set_xlabel('Time (s)')
    ax.set_yticks([])
    ax.set_title(title)
    ax.set_xlim(-2, events_df['onset'].max() + events_df['duration'].max() + 5)
    ax.spines[['left', 'top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


plot_trial_timeline(bids_events)

## 8  Computing Condition Summary Statistics

In [ ]:
# Cell 11: Summary statistics per condition
print('=== Trial counts ===')
print(bids_events['trial_type'].value_counts().sort_index())

print('\n=== Response time summary (s) ===')
rt_summary = (
    bids_events
    .groupby('trial_type')['response_time']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
    .round(3)
)
print(rt_summary)

# Accuracy (if 'correct' column is present)
if 'correct' in bids_events.columns:
    print('\n=== Accuracy (proportion correct) ===')
    acc = bids_events.groupby('trial_type')['correct'].mean().round(3)
    print(acc)

In [ ]:
# Cell 12: Reaction time distribution plot
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
conditions = sorted(bids_events['trial_type'].unique())

for ax, cond in zip(axes, conditions):
    subset = bids_events.loc[bids_events['trial_type'] == cond, 'response_time'].dropna()
    ax.hist(subset, bins=10,
            color=CONDITION_COLORS.get(cond, '#888888'),
            edgecolor='white', alpha=0.85)
    ax.axvline(subset.mean(), color='black', linestyle='--',
               linewidth=1.2, label=f'mean={subset.mean():.2f}s')
    ax.set_title(cond)
    ax.set_xlabel('Response time (s)')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Count')
fig.suptitle('RT Distributions by Condition', fontsize=13)
plt.tight_layout()
plt.show()

## 9  Using the Command-Line Scripts

The `scripts/` directory contains three standalone Python scripts:

```bash
# 1. Convert PsychoPy CSV → BIDS TSV
python scripts/convert_psychopy_to_bids_events.py \
    --input  psychopy_output.csv \
    --output sub-01_task-emotionreg_run-1_events.tsv \
    --task_name emotionreg --run 1

# 2. Validate a BIDS events file
python scripts/validate_events.py \
    --events_file sub-01_task-emotionreg_run-1_events.tsv \
    --bold_json  sub-01_task-emotionreg_run-1_bold.json

# 3. Generate GLM contrast specification
python scripts/create_condition_contrasts.py \
    --events_file sub-01_task-emotionreg_run-1_events.tsv \
    --output contrasts.json
```

Run the validator on the file we just wrote:

In [ ]:
# Cell 13: Call the validate_events script programmatically
scripts_dir = os.path.join(os.getcwd(), 'scripts')
sys.path.insert(0, scripts_dir)

from validate_events import main as validate_main

exit_code = validate_main(['--events_file', out_path])
print(f'\nScript exit code: {exit_code}  (0 = passed)')

## 10  Summary and Next Steps

In this module you learned how to:

- Understand the BIDS events TSV format and required/recommended columns.
- Load and explore existing events files with pandas.
- Parse a PsychoPy-style log file and compute BIDS-relative onset times.
- Write a BIDS-compliant TSV with `n/a` for missing values.
- Validate events files for timing consistency and overlapping trials.
- Visualise trial structure as a timeline (Gantt) plot.
- Compute per-condition summary statistics and RT distributions.

### Next

➡️  **Module 05: Quality Control with MRIQC**  
Run MRIQC on your BIDS dataset to check image quality before preprocessing.

### References

- [BIDS Specification — Task events](https://bids-specification.readthedocs.io/en/stable/modality-specific-files/task-events.html)
- [PsychoPy documentation](https://www.psychopy.org/documentation.html)
- [nilearn first-level GLM](https://nilearn.github.io/stable/glm/first_level_model.html)